# Algorithm 25B: Raw-Native KLDM Lattice CPS

This notebook tests the safer raw-native version:

`KLDM-PC -> clean lattice estimate -> raw length/angle family correction -> soft CPS -> continue KLDM-PC`

This is **not** faithful DiffCSP++ k-space projection. It is a KLDM-native lattice-shape regularizer that avoids the basis mismatch seen for cubic/hexagonal k-space projection.

Laptop defaults:

- tests all loaded graphs for cheap identity diagnostics
- sampler runs only on raw-GT-identity-safe graphs by default
- baseline PC800 is cached from the previous prompt table unless `RAW_NATIVE_RERUN_BASELINE=1`
- raw-native PC800 runs one graph and one seed by default

In [ ]:
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")
os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")

import math
import time
import importlib
from dataclasses import dataclass
from pathlib import Path
from types import SimpleNamespace
from typing import Any

import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

ROOT = Path.cwd().resolve()
if not ((ROOT / 'configs').exists() and (ROOT / 'src').exists()):
    if ((ROOT.parent / 'configs').exists() and (ROOT.parent / 'src').exists()):
        ROOT = ROOT.parent

import kldmPlus.algorithm25_kldm_pc_cps_lattice as kspace_backend
import kldmPlus.algorithm25_raw_native_lattice_cps as raw_backend
kspace_backend = importlib.reload(kspace_backend)
raw_backend = importlib.reload(raw_backend)

from kldmPlus.run_sampling_compare import SamplingCompareRunner, set_seed
from kldmPlus.sample_evaluation import build_structure_from_sample, evaluate_csp_reconstruction as evaluate_csp_reconstruction_plus
from kldm.sample_evaluation import evaluate_csp_reconstruction as evaluate_csp_reconstruction_kldm

CONFIG_PATH = ROOT / 'configs' / 'kldm_plus' / 'mp_20' / 'mp20_sampling_compare_em_vs_algorithm10_casal_chart.yaml'
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)

SAMPLE_SEED = int(os.environ.get('RAW_NATIVE_SEED', '2'))
GRAPH_IDS = [int(v.strip()) for v in os.environ.get('RAW_NATIVE_GRAPH_IDS', '1,2,3').split(',') if v.strip()]
RAW_NATIVE_TOTAL_STEPS = int(os.environ.get('RAW_NATIVE_TOTAL_STEPS', '800'))
RAW_NATIVE_LOCAL_REMAINING_STEP = int(os.environ.get('RAW_NATIVE_LOCAL_REMAINING_STEP', '200'))
RAW_NATIVE_SAMPLER_GRAPH_LIMIT = int(os.environ.get('RAW_NATIVE_SAMPLER_GRAPH_LIMIT', '1'))
RAW_NATIVE_SAMPLER_SEED_LIMIT = int(os.environ.get('RAW_NATIVE_SAMPLER_SEED_LIMIT', '1'))
RAW_NATIVE_RERUN_BASELINE = os.environ.get('RAW_NATIVE_RERUN_BASELINE', '0').strip().lower() in {'1', 'true', 'yes'}
RAW_NATIVE_DEBUG_SAMPLERS = os.environ.get('RAW_NATIVE_DEBUG_SAMPLERS', '1').strip().lower() in {'1', 'true', 'yes'}
RAW_NATIVE_DEBUG_PRINT_EVERY = int(os.environ.get('RAW_NATIVE_DEBUG_PRINT_EVERY', '100'))

RAW_NATIVE_CFG = raw_backend.Algorithm25RawNativeConfig(
    total_steps=RAW_NATIVE_TOTAL_STEPS,
    projection_start_frac=float(os.environ.get('RAW_NATIVE_START_FRAC', '0.25')),
    projection_interval=int(os.environ.get('RAW_NATIVE_INTERVAL', '50')),
    gamma_min=0.05,
    gamma_max=float(os.environ.get('RAW_NATIVE_GAMMA_MAX', '3.0')),
    gamma_power=2.0,
    tau=0.25,
    n_correction_steps=1,
    use_gate=True,
    raw_violation_min=float(os.environ.get('RAW_NATIVE_VIOLATION_MIN', '1e-8')),
    raw_violation_max=float(os.environ.get('RAW_NATIVE_VIOLATION_MAX', '4.0')),
    shift_cap_scaled=float(os.environ.get('RAW_NATIVE_SHIFT_CAP', '1.0')),
    angle_scale_deg=float(os.environ.get('RAW_NATIVE_ANGLE_SCALE_DEG', '10.0')),
    use_final_projection=False,
)

runner = SamplingCompareRunner(config_path=CONFIG_PATH)
runner.model.eval()
set_seed(SAMPLE_SEED)

requested_num_targets = max(max(GRAPH_IDS) if GRAPH_IDS else 0, len(GRAPH_IDS), 5)
if int(runner.compare_cfg.get('num_targets', 0)) < requested_num_targets:
    runner.compare_cfg['num_targets'] = int(requested_num_targets)
    runner.compare_cfg['batch_size'] = max(int(runner.compare_cfg.get('batch_size', 0)), int(requested_num_targets))
    runner.loader, runner.lattice_transform = runner._build_loader()

full_batch = next(iter(runner.loader)).to(runner.device)
full_ptr = full_batch.ptr.tolist()
full_num_graphs = max(len(full_ptr) - 1, 0)
available_graph_ids = [int(graph_id) for graph_id in GRAPH_IDS if 1 <= int(graph_id) <= full_num_graphs]
selected_graph_indices0 = [int(graph_id) - 1 for graph_id in available_graph_ids]
selected_items = full_batch.index_select(selected_graph_indices0) if hasattr(full_batch, 'index_select') else full_batch[selected_graph_indices0]
if isinstance(selected_items, list):
    batch = full_batch.__class__.from_data_list(selected_items)
else:
    batch = selected_items
batch = batch.to(runner.device)
ptr = batch.ptr.detach().cpu().tolist()

RAW_NATIVE_CACHE: dict[tuple[Any, ...], Any] = {}
result_tables: dict[str, pd.DataFrame] = {}
print(f'raw_native graphs={available_graph_ids} seed={SAMPLE_SEED} total_steps={RAW_NATIVE_TOTAL_STEPS} cfg={RAW_NATIVE_CFG}')

In [ ]:
@dataclass
class GraphCase:
    graph_id: int
    graph_idx0: int
    composition: dict[int, int]
    atomic_numbers: torch.Tensor
    gt_frac: torch.Tensor
    gt_l_feature: torch.Tensor
    requested_sg: int


def dataframe_result(name: str, rows: list[dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    if 'PASS' not in df.columns:
        df['PASS'] = False
    if 'status' not in df.columns:
        df['status'] = np.where(df['PASS'], 'PASS', 'FAIL')
    result_tables[name] = df
    return df


def safe_display_sorted(df: pd.DataFrame, by: list[str]):
    df = df.copy()
    for key in by:
        if key not in df.columns:
            df[key] = np.nan
    display(df.sort_values(by).reset_index(drop=True))


def safe_metric_float(value) -> float:
    if value is None:
        return float('nan')
    if torch.is_tensor(value):
        value = value.detach().reshape(-1)
        if value.numel() == 0:
            return float('nan')
        value = value[0].item()
    try:
        return float(value)
    except Exception:
        return float('nan')


def safe_metric_bool(value) -> bool:
    if torch.is_tensor(value):
        value = value.detach().reshape(-1)
        if value.numel() == 0:
            return False
        value = value[0].item()
    return bool(value)


def finite_mean(values) -> float:
    vals = [float(v) for v in values if np.isfinite(float(v))]
    return float(np.mean(vals)) if vals else float('nan')


def graph_slice(graph_idx0: int) -> tuple[int, int]:
    return int(ptr[graph_idx0]), int(ptr[graph_idx0 + 1])


def composition_counter(values) -> dict[int, int]:
    from collections import Counter
    arr = [int(v) for v in torch.as_tensor(values, dtype=torch.long).reshape(-1).tolist()]
    return dict(sorted(Counter(arr).items()))


def graph_edge_node_index(case: GraphCase) -> torch.Tensor:
    start, end = graph_slice(case.graph_idx0)
    edge_index = batch.edge_node_index
    mask = ((edge_index[0] >= start) & (edge_index[0] < end) & (edge_index[1] >= start) & (edge_index[1] < end))
    return (edge_index[:, mask] - start).detach().clone()


def graph_tensors(graph_idx0: int) -> dict[str, Any]:
    start, end = graph_slice(graph_idx0)
    return {
        'pos': batch.pos[start:end].detach().clone(),
        'l': batch.l[graph_idx0].detach().clone().reshape(-1),
        'h': batch.atomic_numbers[start:end].detach().clone().to(torch.long),
        'sg': int(torch.as_tensor(batch.space_group).reshape(-1)[graph_idx0].item()),
    }


def load_test_graphs(graph_ids=available_graph_ids) -> list[GraphCase]:
    out = []
    for graph_idx0, graph_id in enumerate(graph_ids):
        g = graph_tensors(graph_idx0)
        out.append(GraphCase(
            graph_id=int(graph_id),
            graph_idx0=int(graph_idx0),
            composition=composition_counter(g['h']),
            atomic_numbers=g['h'],
            gt_frac=g['pos'],
            gt_l_feature=g['l'],
            requested_sg=g['sg'],
        ))
    return out


GRAPH_CASES = load_test_graphs(available_graph_ids)
print('loaded_graphs=', [g.graph_id for g in GRAPH_CASES], 'sg=', [g.requested_sg for g in GRAPH_CASES])


def make_single_graph_batch_view(case: GraphCase, *, ref_tensor: torch.Tensor | None = None):
    device = case.gt_frac.device if ref_tensor is None else ref_tensor.device
    dtype = case.gt_frac.dtype if ref_tensor is None else ref_tensor.dtype
    pos = case.gt_frac.detach().clone().to(device=device, dtype=dtype)
    l = case.gt_l_feature.detach().clone().reshape(1, -1).to(device=device, dtype=case.gt_l_feature.dtype)
    atomic_numbers = case.atomic_numbers.detach().clone().to(device=device, dtype=torch.long)
    batch_index = torch.zeros(pos.shape[0], device=device, dtype=torch.long)
    num_atoms = torch.tensor([int(pos.shape[0])], device=device, dtype=torch.long)
    edge_node_index = graph_edge_node_index(case).to(device=device)
    space_group = torch.tensor([int(case.requested_sg)], device=device, dtype=torch.long)

    class _SingleGraphBatch(SimpleNamespace):
        def to(self, device):
            out = _SingleGraphBatch()
            for key, value in self.__dict__.items():
                setattr(out, key, value.to(device) if torch.is_tensor(value) else value)
            return out

    return _SingleGraphBatch(
        pos=pos,
        l=l,
        atomic_numbers=atomic_numbers,
        batch=batch_index,
        num_atoms=num_atoms,
        num_graphs=1,
        edge_node_index=edge_node_index,
        space_group=space_group,
    )


def _cache_key(*parts: Any) -> tuple[Any, ...]:
    return tuple(parts)


def _cached(key: tuple[Any, ...], fn):
    if key not in RAW_NATIVE_CACHE:
        RAW_NATIVE_CACHE[key] = fn()
    return RAW_NATIVE_CACHE[key]

In [ ]:
def cell_lengths_angles(cell: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    cell = torch.as_tensor(cell).reshape(3, 3)
    lengths = torch.linalg.norm(cell, dim=1)
    def angle(u, v):
        denom = (torch.linalg.norm(u) * torch.linalg.norm(v)).clamp_min(1.0e-12)
        cosine = torch.clamp(torch.dot(u, v) / denom, -1.0, 1.0)
        return torch.rad2deg(torch.acos(cosine))
    alpha = angle(cell[1], cell[2])
    beta = angle(cell[0], cell[2])
    gamma = angle(cell[0], cell[1])
    return lengths, torch.stack([alpha, beta, gamma])


def cell_volume(cell: torch.Tensor) -> float:
    return float(torch.abs(torch.det(torch.as_tensor(cell).reshape(3, 3))).detach().item())


def lattice_metric_dict(pred_l: torch.Tensor, target_l: torch.Tensor, case: GraphCase) -> dict[str, float]:
    num_atoms = int(case.atomic_numbers.shape[0])
    pred_cell = kspace_backend.lattice6_to_matrix(pred_l, num_atoms=num_atoms, lattice_transform=runner.lattice_transform)
    gt_cell = kspace_backend.lattice6_to_matrix(target_l, num_atoms=num_atoms, lattice_transform=runner.lattice_transform)
    pred_lengths, pred_angles = cell_lengths_angles(pred_cell)
    gt_lengths, gt_angles = cell_lengths_angles(gt_cell)
    pred_volume = cell_volume(pred_cell)
    gt_volume = cell_volume(gt_cell)
    return {
        'cell_matrix_rmse': float(torch.sqrt(torch.mean((pred_cell - gt_cell) ** 2)).detach().item()),
        'length_rmse': float(torch.sqrt(torch.mean((pred_lengths - gt_lengths) ** 2)).detach().item()),
        'angle_rmse_deg': float(torch.sqrt(torch.mean((pred_angles - gt_angles) ** 2)).detach().item()),
        'volume_abs_error': abs(pred_volume - gt_volume),
        'volume_rel_error': abs(pred_volume - gt_volume) / max(abs(gt_volume), 1.0e-12),
    }


def build_case_structure(case: GraphCase):
    return build_structure_from_sample(case.gt_frac, case.gt_l_feature, case.atomic_numbers, lattice_transform=runner.lattice_transform)


def detect_spacegroup_family(pred_f: torch.Tensor, pred_l: torch.Tensor, pred_a: torch.Tensor) -> dict[str, Any]:
    try:
        structure = build_structure_from_sample(pred_f, pred_l, pred_a, lattice_transform=runner.lattice_transform)
        sga = SpacegroupAnalyzer(structure, symprec=1.0e-1, angle_tolerance=5.0)
        sg = int(sga.get_space_group_number())
        return {'detected_space_group': sg, 'detected_family': kspace_backend.spacegroup_to_crystal_family(sg)}
    except Exception:
        return {'detected_space_group': None, 'detected_family': None}


def evaluate_arrays(case: GraphCase, pred_f: torch.Tensor, pred_l: torch.Tensor, pred_h: torch.Tensor) -> dict[str, Any]:
    plus = evaluate_csp_reconstruction_plus(
        pred_f=pred_f,
        pred_l=pred_l,
        pred_a=pred_h.to(torch.long),
        target_f=case.gt_frac,
        target_l=case.gt_l_feature,
        target_a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
        requested_space_group=case.requested_sg,
        validity_cutoff=0.5,
    )
    kldm = evaluate_csp_reconstruction_kldm(
        pred_f=pred_f,
        pred_l=pred_l,
        pred_a=pred_h.to(torch.long),
        target_f=case.gt_frac,
        target_l=case.gt_l_feature,
        target_a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
    )
    return {
        'match': bool(kldm.match),
        'valid': bool(kldm.valid),
        'rmse': kldm.rmse,
        'frac_rmse': plus.frac_rmse,
        'plus_match': bool(plus.match),
        'plus_valid': bool(plus.valid),
        'plus_rmse': plus.rmse,
        'requested_space_group_match': plus.requested_space_group_match,
    }


def raw_violation_for_lattice(case: GraphCase, pred_l: torch.Tensor) -> float:
    _, diag = raw_backend.raw_native_project_lattice6(
        pred_l.reshape(-1),
        num_atoms=int(case.atomic_numbers.shape[0]),
        lattice_transform=runner.lattice_transform,
        space_group_number=int(case.requested_sg),
        angle_scale_deg=float(RAW_NATIVE_CFG.angle_scale_deg),
        shift_cap_scaled=None,
    )
    return safe_metric_float(diag.raw_violation_before)


def final_sampler_metrics(case: GraphCase, pred_f: torch.Tensor, pred_l: torch.Tensor, pred_a: torch.Tensor) -> dict[str, Any]:
    eval_out = evaluate_arrays(case, pred_f, pred_l.reshape(-1), pred_a)
    lattice_metrics = lattice_metric_dict(pred_l.reshape(-1), case.gt_l_feature, case)
    sym = detect_spacegroup_family(pred_f, pred_l.reshape(-1), pred_a)
    raw_violation = raw_violation_for_lattice(case, pred_l.reshape(-1))
    return {'eval': eval_out, 'lattice_metrics': lattice_metrics, 'symmetry': sym, 'final_raw_violation': raw_violation}

### Test A: Raw-Native GT Identity

Project the GT lattice directly in KLDM's native decoded length/angle basis.

If this moves GT a lot, raw-native CPS is not safe for that graph/family unless used only as a diagnostic or very weak late correction.

In [ ]:
rows_a = []
for case in GRAPH_CASES:
    ell_raw, diag = raw_backend.raw_native_project_lattice6(
        case.gt_l_feature.reshape(-1),
        num_atoms=int(case.atomic_numbers.shape[0]),
        lattice_transform=runner.lattice_transform,
        space_group_number=int(case.requested_sg),
        angle_scale_deg=float(RAW_NATIVE_CFG.angle_scale_deg),
        shift_cap_scaled=None,
    )
    metrics = lattice_metric_dict(ell_raw, case.gt_l_feature, case)
    rows_a.append({
        'test': 'algorithm25b_a_raw_native_gt_identity',
        'graph': case.graph_id,
        'space_group': int(case.requested_sg),
        'family': diag.family,
        'selected_variant': diag.selected_variant,
        **metrics,
        'raw_violation_before': safe_metric_float(diag.raw_violation_before),
        'raw_violation_after': safe_metric_float(diag.raw_violation_after),
        'use_for_sampler': bool(metrics['length_rmse'] < 1e-3 and metrics['angle_rmse_deg'] < 1e-2),
        'PASS': bool(metrics['length_rmse'] < 1e-3 and metrics['angle_rmse_deg'] < 1e-2),
        'status': 'INFO',
    })

safe_display_sorted(dataframe_result('algorithm25b_a_raw_native_gt_identity', rows_a), ['graph'])

### Test B: Clean Estimate Projection

Use one cheap GT-noisy local state, predict the clean lattice, then apply raw-native projection to the clean lattice estimate only.

This is not a sampler test; it checks whether the raw-native projection moves the clean lattice in a reasonable direction.

In [ ]:
def local_noisy_state(case: GraphCase, *, remaining_step: int, seed: int = SAMPLE_SEED) -> dict[str, Any]:
    tau = float(remaining_step) / float(max(RAW_NATIVE_TOTAL_STEPS, 1))
    batch_view = make_single_graph_batch_view(case, ref_tensor=case.gt_frac)
    set_seed(int(seed) + 1000 * int(case.graph_id) + int(remaining_step))
    t_graph = torch.as_tensor([[tau]], device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    t_nodes = torch.full((int(case.gt_frac.shape[0]),), tau, device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    f_t, v_t, *_ = runner.model.tdm.sample_noisy_state(t=t_nodes, f0=case.gt_frac, index=batch_view.batch)
    t_lattice = t_graph.reshape(-1)
    set_seed(int(seed) + 2000 * int(case.graph_id) + int(remaining_step))
    l_t, _ = runner.model.diffusion_l.forward_sample(
        t=t_lattice,
        x0=case.gt_l_feature.reshape(1, -1),
        num_atoms=batch_view.num_atoms,
    )
    return {
        'f_t': f_t,
        'v_t': v_t,
        'l_t': l_t.reshape(1, -1),
        'a_t': case.atomic_numbers,
        'node_index': batch_view.batch,
        'edge_node_index': batch_view.edge_node_index,
        't_graph': t_graph,
        't_nodes': t_nodes,
        't_lattice': t_lattice,
        'batch_view': batch_view,
    }


rows_b = []
for case in GRAPH_CASES[:1]:
    state = local_noisy_state(case, remaining_step=RAW_NATIVE_LOCAL_REMAINING_STEP, seed=SAMPLE_SEED)
    preds = runner.model.score_network(
        t=state['t_graph'],
        pos=state['f_t'],
        v=state['v_t'],
        h=state['a_t'],
        l=state['l_t'],
        node_index=state['node_index'],
        edge_node_index=state['edge_node_index'],
    )
    ell0_hat = kspace_backend.predict_clean_lattice_from_prediction(
        l_t=state['l_t'],
        pred_l=preds['l'],
        t_lattice=state['t_lattice'],
        diffusion_l=runner.model.diffusion_l,
        num_atoms=state['batch_view'].num_atoms,
    ).reshape(-1)
    ell0_soft, diag = raw_backend.raw_native_cps_project_clean_estimate(
        ell0_hat,
        num_atoms=int(case.atomic_numbers.shape[0]),
        lattice_transform=runner.lattice_transform,
        space_group_number=int(case.requested_sg),
        tau=float(RAW_NATIVE_LOCAL_REMAINING_STEP) / float(RAW_NATIVE_TOTAL_STEPS),
        gamma_min=float(RAW_NATIVE_CFG.gamma_min),
        gamma_max=float(RAW_NATIVE_CFG.gamma_max),
        gamma_power=float(RAW_NATIVE_CFG.gamma_power),
        projection_start_frac=float(RAW_NATIVE_CFG.projection_start_frac),
        angle_scale_deg=float(RAW_NATIVE_CFG.angle_scale_deg),
        use_gate=bool(RAW_NATIVE_CFG.use_gate),
        raw_violation_min=float(RAW_NATIVE_CFG.raw_violation_min),
        raw_violation_max=float(RAW_NATIVE_CFG.raw_violation_max),
        shift_cap_scaled=float(RAW_NATIVE_CFG.shift_cap_scaled),
    )
    before = lattice_metric_dict(ell0_hat, case.gt_l_feature, case)
    after = lattice_metric_dict(ell0_soft, case.gt_l_feature, case)
    rows_b.append({
        'test': 'algorithm25b_b_clean_estimate_projection',
        'graph': case.graph_id,
        'remaining_step': int(RAW_NATIVE_LOCAL_REMAINING_STEP),
        'tau': float(RAW_NATIVE_LOCAL_REMAINING_STEP) / float(RAW_NATIVE_TOTAL_STEPS),
        'family': diag.family,
        'selected_variant': diag.selected_variant,
        'accepted': bool(diag.accepted),
        'skipped_reason': diag.skipped_reason,
        'weight': safe_metric_float(diag.weight),
        'raw_violation_before': safe_metric_float(diag.raw_violation_before),
        'raw_violation_after': safe_metric_float(diag.raw_violation_after),
        'clean_lattice_rmse_before': before['cell_matrix_rmse'],
        'clean_lattice_rmse_after': after['cell_matrix_rmse'],
        'length_rmse_before': before['length_rmse'],
        'length_rmse_after': after['length_rmse'],
        'angle_rmse_before_deg': before['angle_rmse_deg'],
        'angle_rmse_after_deg': after['angle_rmse_deg'],
        'PASS': bool(diag.accepted or diag.skipped_reason == 'below_min_violation'),
        'status': 'INFO',
    })

safe_display_sorted(dataframe_result('algorithm25b_b_clean_estimate_projection', rows_b), ['graph', 'remaining_step'])

### Test C: One-Step State Stability

Apply one raw-native CPS lattice update to the current noisy lattice state, then query KLDM again.

This checks whether the update destabilizes the next clean lattice prediction.

In [ ]:
rows_c = []
for case in GRAPH_CASES[:1]:
    state = local_noisy_state(case, remaining_step=RAW_NATIVE_LOCAL_REMAINING_STEP, seed=SAMPLE_SEED)
    preds_before = runner.model.score_network(
        t=state['t_graph'],
        pos=state['f_t'],
        v=state['v_t'],
        h=state['a_t'],
        l=state['l_t'],
        node_index=state['node_index'],
        edge_node_index=state['edge_node_index'],
    )
    ell0_hat = kspace_backend.predict_clean_lattice_from_prediction(
        l_t=state['l_t'],
        pred_l=preds_before['l'],
        t_lattice=state['t_lattice'],
        diffusion_l=runner.model.diffusion_l,
        num_atoms=state['batch_view'].num_atoms,
    ).reshape(-1)
    ell0_soft, proj_diag = raw_backend.raw_native_cps_project_clean_estimate(
        ell0_hat,
        num_atoms=int(case.atomic_numbers.shape[0]),
        lattice_transform=runner.lattice_transform,
        space_group_number=int(case.requested_sg),
        tau=float(RAW_NATIVE_LOCAL_REMAINING_STEP) / float(RAW_NATIVE_TOTAL_STEPS),
        gamma_min=float(RAW_NATIVE_CFG.gamma_min),
        gamma_max=float(RAW_NATIVE_CFG.gamma_max),
        gamma_power=float(RAW_NATIVE_CFG.gamma_power),
        projection_start_frac=float(RAW_NATIVE_CFG.projection_start_frac),
        angle_scale_deg=float(RAW_NATIVE_CFG.angle_scale_deg),
        use_gate=bool(RAW_NATIVE_CFG.use_gate),
        raw_violation_min=float(RAW_NATIVE_CFG.raw_violation_min),
        raw_violation_max=float(RAW_NATIVE_CFG.raw_violation_max),
        shift_cap_scaled=float(RAW_NATIVE_CFG.shift_cap_scaled),
    )
    if proj_diag.accepted:
        l_after, update_diag = kspace_backend.apply_lattice_cps_to_state(
            ell_t=state['l_t'],
            ell0_hat=ell0_hat,
            ell0_soft=ell0_soft,
            t_lattice=state['t_lattice'],
            diffusion_l=runner.model.diffusion_l,
        )
    else:
        l_after = state['l_t']
        update_diag = SimpleNamespace(lattice_state_shift_norm=0.0)

    preds_after = runner.model.score_network(
        t=state['t_graph'],
        pos=state['f_t'],
        v=state['v_t'],
        h=state['a_t'],
        l=l_after.reshape(1, -1),
        node_index=state['node_index'],
        edge_node_index=state['edge_node_index'],
    )
    ell0_after = kspace_backend.predict_clean_lattice_from_prediction(
        l_t=l_after.reshape(1, -1),
        pred_l=preds_after['l'],
        t_lattice=state['t_lattice'],
        diffusion_l=runner.model.diffusion_l,
        num_atoms=state['batch_view'].num_atoms,
    ).reshape(-1)
    before = lattice_metric_dict(ell0_hat, case.gt_l_feature, case)
    after = lattice_metric_dict(ell0_after, case.gt_l_feature, case)
    rows_c.append({
        'test': 'algorithm25b_c_one_step_state_stability',
        'graph': case.graph_id,
        'remaining_step': int(RAW_NATIVE_LOCAL_REMAINING_STEP),
        'accepted': bool(proj_diag.accepted),
        'state_shift_norm': safe_metric_float(update_diag.lattice_state_shift_norm),
        'raw_violation_before': safe_metric_float(proj_diag.raw_violation_before),
        'raw_violation_after': safe_metric_float(proj_diag.raw_violation_after),
        'clean_lattice_rmse_before': before['cell_matrix_rmse'],
        'clean_lattice_rmse_after_state_update': after['cell_matrix_rmse'],
        'length_rmse_before': before['length_rmse'],
        'length_rmse_after_state_update': after['length_rmse'],
        'angle_rmse_before_deg': before['angle_rmse_deg'],
        'angle_rmse_after_state_update': after['angle_rmse_deg'],
        'PASS': bool(after['cell_matrix_rmse'] < before['cell_matrix_rmse'] * 5.0 or not proj_diag.accepted),
        'status': 'INFO',
    })

safe_display_sorted(dataframe_result('algorithm25b_c_one_step_state_stability', rows_c), ['graph', 'remaining_step'])

### Test D: Full Sampler Ablation

Compare cached facitKLDM PC800 baseline against raw-native PC800 CPS.

By default this only runs on graphs that pass Test A, because raw-native projection is supposed to be same-basis. Set environment variables to widen it.

In [ ]:
BASELINE_PC1000_METRIC_CACHE: dict[tuple[int, int], dict[str, Any]] = {
    (1, 2): {'rmse': 0.002905, 'match': True, 'valid': True, 'lattice_rmse': 0.053792, 'length_rmse': 0.068597, 'angle_rmse_deg': 0.628667, 'final_raw_violation': float('nan'), 'detected_space_group': 12, 'detected_family': 'monoclinic', 'runtime_s': 64.487841},
    (1, 3): {'rmse': 0.001229, 'match': True, 'valid': True, 'lattice_rmse': 0.040293, 'length_rmse': 0.035223, 'angle_rmse_deg': 0.646725, 'final_raw_violation': float('nan'), 'detected_space_group': 12, 'detected_family': 'monoclinic', 'runtime_s': 63.958167},
    (2, 2): {'rmse': 0.017590, 'match': True, 'valid': True, 'lattice_rmse': 0.076113, 'length_rmse': 0.123589, 'angle_rmse_deg': 0.407479, 'final_raw_violation': float('nan'), 'detected_space_group': 4, 'detected_family': 'monoclinic', 'runtime_s': 209.335362},
    (2, 3): {'rmse': float('nan'), 'match': False, 'valid': True, 'lattice_rmse': 0.052773, 'length_rmse': 0.061662, 'angle_rmse_deg': 0.612761, 'final_raw_violation': float('nan'), 'detected_space_group': 2, 'detected_family': 'triclinic', 'runtime_s': 1320.370464},
    (3, 2): {'rmse': float('nan'), 'match': False, 'valid': True, 'lattice_rmse': 2.891696, 'length_rmse': 4.047502, 'angle_rmse_deg': 23.863564, 'final_raw_violation': float('nan'), 'detected_space_group': 38, 'detected_family': 'orthorhombic', 'runtime_s': 345.208223},
    (3, 3): {'rmse': float('nan'), 'match': False, 'valid': True, 'lattice_rmse': 4.304510, 'length_rmse': 6.704825, 'angle_rmse_deg': 23.873652, 'final_raw_violation': float('nan'), 'detected_space_group': 38, 'detected_family': 'orthorhombic', 'runtime_s': 178.202295},
}


def cached_baseline_pc1000_metrics(case: GraphCase, seed: int) -> dict[str, Any]:
    cached = dict(BASELINE_PC1000_METRIC_CACHE.get((int(case.graph_id), int(seed)), {}))
    if not cached:
        raise KeyError(f'No cached baseline for graph={case.graph_id}, seed={seed}. Set RAW_NATIVE_RERUN_BASELINE=1.')
    return {
        'f': torch.full_like(case.gt_frac, float('nan')),
        'v': torch.full_like(case.gt_frac, float('nan')),
        'l': torch.full_like(case.gt_l_feature.reshape(-1), float('nan')),
        'a': case.atomic_numbers.detach().clone(),
        'eval': {
            'rmse': cached['rmse'],
            'match': cached['match'],
            'valid': cached['valid'],
            'frac_rmse': float('nan'),
            'plus_rmse': float('nan'),
            'plus_match': cached['match'],
            'plus_valid': cached['valid'],
        },
        'lattice_metrics': {
            'cell_matrix_rmse': cached['lattice_rmse'],
            'length_rmse': cached['length_rmse'],
            'angle_rmse_deg': cached['angle_rmse_deg'],
            'volume_abs_error': float('nan'),
            'volume_rel_error': float('nan'),
        },
        'symmetry': {'detected_space_group': cached['detected_space_group'], 'detected_family': cached['detected_family']},
        'runtime_s': cached['runtime_s'],
        'final_raw_violation': cached['final_raw_violation'],
        'baseline_cache_source': 'user_prompt_table_cache',
    }


def run_baseline_pc1000(case: GraphCase, *, seed: int):
    key = _cache_key('baseline_pc1000', case.graph_id, seed, RAW_NATIVE_TOTAL_STEPS, RAW_NATIVE_RERUN_BASELINE)
    if not RAW_NATIVE_RERUN_BASELINE and RAW_NATIVE_TOTAL_STEPS == 1000:
        return cached_baseline_pc1000_metrics(case, int(seed))

    def _runner():
        batch_view = make_single_graph_batch_view(case, ref_tensor=case.gt_frac)
        set_seed(int(seed) + 920000 * int(case.graph_id) + int(RAW_NATIVE_TOTAL_STEPS))
        t0 = time.perf_counter()
        f_t, v_t, l_t, a_t = runner.model.sample_CSP_algorithm4_facit(
            n_steps=int(RAW_NATIVE_TOTAL_STEPS),
            batch=batch_view,
            t_start=1.0,
            t_final=1.0e-6,
            tau=float(RAW_NATIVE_CFG.tau),
            n_correction_steps=int(RAW_NATIVE_CFG.n_correction_steps),
            debug_label=f'raw_native_baseline_g{case.graph_id}_s{seed}' if RAW_NATIVE_DEBUG_SAMPLERS else None,
            debug_print_every=RAW_NATIVE_DEBUG_PRINT_EVERY if RAW_NATIVE_DEBUG_SAMPLERS else None,
        )
        runtime_s = time.perf_counter() - t0
        metrics = final_sampler_metrics(case, f_t, l_t.reshape(-1), a_t)
        return {'f': f_t, 'v': v_t, 'l': l_t.reshape(-1), 'a': a_t, 'runtime_s': runtime_s, **metrics}
    return _cached(key, _runner)


def run_raw_native_sampler(case: GraphCase, *, seed: int, config: raw_backend.Algorithm25RawNativeConfig, label: str):
    key = _cache_key('raw_native_sampler', case.graph_id, seed, label, config)
    if RAW_NATIVE_DEBUG_SAMPLERS:
        print(f'[raw-native-notebook] {"cache-hit" if key in RAW_NATIVE_CACHE else "start"} {label} graph={case.graph_id} seed={seed}', flush=True)

    def _runner():
        batch_view = make_single_graph_batch_view(case, ref_tensor=case.gt_frac)
        set_seed(int(seed) + 930000 * int(case.graph_id) + abs(hash(label)) % 10000)
        t0 = time.perf_counter()
        result = raw_backend.kldm_pc_cps_raw_native_lattice_sampler(
            model=runner.model,
            batch=batch_view,
            lattice_transform=runner.lattice_transform,
            oracle_space_group=int(case.requested_sg),
            config=config,
            t_start=1.0,
            t_final=1.0e-6,
            debug_label=label if RAW_NATIVE_DEBUG_SAMPLERS else None,
            debug_print_every=RAW_NATIVE_DEBUG_PRINT_EVERY if RAW_NATIVE_DEBUG_SAMPLERS else None,
        )
        runtime_s = time.perf_counter() - t0
        metrics = final_sampler_metrics(case, result.frac_coords, result.lattice.reshape(-1), result.atom_types)
        return {'result': result, 'runtime_s': runtime_s, **metrics}
    return _cached(key, _runner)


def expected_projection_count(config: raw_backend.Algorithm25RawNativeConfig) -> int:
    return sum(
        int(kspace_backend.should_project_step(
            remaining_step=int(remaining),
            total_steps=int(config.total_steps),
            projection_start_frac=float(config.projection_start_frac),
            projection_interval=int(config.projection_interval),
        ))
        for remaining in range(int(config.total_steps), 0, -1)
    )


def sampler_row(*, case: GraphCase, seed: int, method: str, result: dict[str, Any], config=None) -> dict[str, Any]:
    ev = result['eval']
    lattice_metrics = result['lattice_metrics']
    interventions = tuple(getattr(result.get('result', SimpleNamespace(interventions=())), 'interventions', ()))
    return {
        'test': 'algorithm25b_d_sampler_compare',
        'graph': case.graph_id,
        'seed': int(seed),
        'method': method,
        'actual_space_group': int(case.requested_sg),
        'requested_space_group': int(case.requested_sg),
        'rmse': safe_metric_float(ev.get('rmse')),
        'match': safe_metric_bool(ev.get('match')),
        'valid': safe_metric_bool(ev.get('valid')),
        'frac_rmse': safe_metric_float(ev.get('frac_rmse')),
        'lattice_rmse': lattice_metrics.get('cell_matrix_rmse', float('nan')),
        'length_rmse': lattice_metrics.get('length_rmse', float('nan')),
        'angle_rmse_deg': lattice_metrics.get('angle_rmse_deg', float('nan')),
        'final_raw_violation': safe_metric_float(result['final_raw_violation']),
        'detected_space_group': result['symmetry']['detected_space_group'],
        'detected_family': result['symmetry']['detected_family'],
        'num_interventions': int(len(interventions)),
        'expected_projection_steps': expected_projection_count(config) if config is not None else 0,
        'runtime_s': safe_metric_float(result['runtime_s']),
        'baseline_cache_source': result.get('baseline_cache_source', ''),
        'PASS': True,
        'status': 'INFO',
    }


def sampler_cases() -> list[GraphCase]:
    identity = result_tables.get('algorithm25b_a_raw_native_gt_identity', pd.DataFrame())
    if len(identity) and 'use_for_sampler' in identity.columns:
        safe_ids = set(int(v) for v in identity.loc[identity['use_for_sampler'] == True, 'graph'].tolist())
        safe_cases = [case for case in GRAPH_CASES if int(case.graph_id) in safe_ids]
        if safe_cases:
            return safe_cases[:RAW_NATIVE_SAMPLER_GRAPH_LIMIT]
    graph2 = [case for case in GRAPH_CASES if int(case.graph_id) == 2]
    return (graph2 or GRAPH_CASES)[:RAW_NATIVE_SAMPLER_GRAPH_LIMIT]


rows_d = []
for case in sampler_cases():
    for seed in [SAMPLE_SEED, SAMPLE_SEED + 1][:RAW_NATIVE_SAMPLER_SEED_LIMIT]:
        baseline = run_baseline_pc1000(case, seed=int(seed))
        rows_d.append(sampler_row(case=case, seed=int(seed), method=f'baseline_pc{RAW_NATIVE_TOTAL_STEPS}', result=baseline, config=None))
        label = f'raw_native_pc{RAW_NATIVE_TOTAL_STEPS}_sf{RAW_NATIVE_CFG.projection_start_frac:.2f}_i{RAW_NATIVE_CFG.projection_interval}_g{RAW_NATIVE_CFG.gamma_max:.0f}'
        out = run_raw_native_sampler(case, seed=int(seed), config=RAW_NATIVE_CFG, label=label)
        rows_d.append(sampler_row(case=case, seed=int(seed), method=label, result=out, config=RAW_NATIVE_CFG))

sampler_df = dataframe_result('algorithm25b_d_sampler_compare', rows_d)
safe_display_sorted(sampler_df, ['graph', 'seed', 'method'])

summary_cols = ['method']
metric_cols = ['rmse', 'match', 'valid', 'frac_rmse', 'lattice_rmse', 'length_rmse', 'angle_rmse_deg', 'final_raw_violation', 'num_interventions', 'runtime_s']
rows_summary = []
for method, group in sampler_df.groupby(summary_cols, dropna=False):
    method_value = method[0] if isinstance(method, tuple) else method
    row = {'test': 'algorithm25b_d_sampler_summary', 'method': method_value, 'num_runs': int(len(group)), 'PASS': True, 'status': 'INFO'}
    for col in metric_cols:
        vals = pd.to_numeric(group[col], errors='coerce') if col in group.columns else pd.Series(dtype=float)
        row[f'mean_{col}'] = float(vals.mean()) if vals.notna().any() else float('nan')
    rows_summary.append(row)

safe_display_sorted(dataframe_result('algorithm25b_d_sampler_summary', rows_summary), ['method'])

### Verdict

Interpretation guide:

- If Test A fails for a graph, raw-native projection is not same-basis safe for that graph.
- If Test B improves raw violation but worsens lattice RMSE badly, projection is enforcing family shape rather than GT lattice accuracy.
- If Test C destabilizes the next denoiser prediction, lower `gamma_max`, start later, or tighten `shift_cap_scaled`.
- If Test D improves match/valid/RMSE/lattice metrics, raw-native CPS is a useful KLDM-native regularizer.

In [ ]:
rows_v = []
test_a = result_tables.get('algorithm25b_a_raw_native_gt_identity', pd.DataFrame())
test_b = result_tables.get('algorithm25b_b_clean_estimate_projection', pd.DataFrame())
test_c = result_tables.get('algorithm25b_c_one_step_state_stability', pd.DataFrame())
test_d = result_tables.get('algorithm25b_d_sampler_compare', pd.DataFrame())

identity_safe = bool(len(test_a) and test_a['use_for_sampler'].any())
clean_projection_ok = bool(len(test_b) and test_b['PASS'].all())
one_step_ok = bool(len(test_c) and test_c['PASS'].all())
sampler_has_cps = bool(len(test_d) and np.any(test_d['method'].astype(str).str.contains('raw_native')))

if identity_safe and clean_projection_ok and one_step_ok and sampler_has_cps:
    recommendation = 'raw_native_ready_for_small_multi_seed_ablation'
elif identity_safe and clean_projection_ok:
    recommendation = 'local_positive_run_more_sampler_debug'
elif identity_safe:
    recommendation = 'identity_safe_but_projection_or_state_update_needs_tuning'
else:
    recommendation = 'do_not_use_raw_native_sampler_on_current_cells_without_basis_gate'

rows_v.append({
    'test': 'algorithm25b_verdict',
    'identity_safe_graph_exists': identity_safe,
    'clean_projection_ok': clean_projection_ok,
    'one_step_ok': one_step_ok,
    'sampler_has_cps_result': sampler_has_cps,
    'recommendation': recommendation,
    'PASS': bool(identity_safe),
    'status': 'INFO',
})

safe_display_sorted(dataframe_result('algorithm25b_verdict', rows_v), ['test'])